In [ ]:
# BAYESIAN NETWORK

# ================== IMPORTS ==================
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination


# ================== STEP 1: DEFINE STRUCTURE ==================
# Example structure (change based on question)

model = BayesianNetwork([
    ('A', 'C'),   # A → C
    ('B', 'C')    # B → C
])


# ============================================================
#  STEP 2: DEFINE CPDs (MOST IMPORTANT PART)
# ============================================================

# ------------------ RULE 1: ROOT NODE (NO PARENTS) ------------------
# Just simple probabilities

cpd_A = TabularCPD(
    variable='A', variable_card=2,
    values=[[0.6],   # A = 0
            [0.4]]   # A = 1
)

cpd_B = TabularCPD(
    variable='B', variable_card=2,
    values=[[0.7],
            [0.3]]
)


# ------------------ RULE 2: NODE WITH PARENTS ------------------
# Example: C depends on A and B

# STEP 1: count parent combinations
# A has 2 values, B has 2 values → 2 × 2 = 4 combinations

# STEP 2: each column = one parent combination
# STEP 3: each row = probability of C states

cpd_C = TabularCPD(
    variable='C',
    variable_card=2,

    values=[
        # C = 0 (No)
        [0.9, 0.6, 0.7, 0.1],

        # C = 1 (Yes)
        [0.1, 0.4, 0.3, 0.9]
    ],

    evidence=['A', 'B'],
    evidence_card=[2, 2]
)


# ============================================================
#  HOW TO DEFINE CPD FOR PARENT NODES (VERY IMPORTANT RULE)
# ============================================================

"""
RULES:

1. Identify parent nodes
   Example: C has parents A and B

2. Find total combinations:
   multiply number of states of each parent
   → A(2) × B(2) = 4 columns

3. Each column = one combination:
   (A,B):
   (0,0), (0,1), (1,0), (1,1)

4. Each column must sum to 1

5. Each row = state of child node
"""


# ================== STEP 3: ADD CPDs ==================
model.add_cpds(cpd_A, cpd_B, cpd_C)

model.check_model()


# ================== STEP 4: INFERENCE ==================
infer = VariableElimination(model)

result = infer.query(
    variables=['C'],
    evidence={'A': 1, 'B': 0}
)

print(result)

In [ ]:
# MARKOV MODEL...next state depends on current state only

import numpy as np

# ================== STATES ==================
# State 0 = Sunny
# State 1 = Rainy

states = ["Sunny", "Rainy"]

# ================== TRANSITION MATRIX ==================
# Rows = current state
# Columns = next state

transition_matrix = np.array([
    # Sunny →   Sunny   Rainy
    [0.8,       0.2],   # From Sunny

    # Rainy →   Sunny   Rainy
    [0.4,       0.6]    # From Rainy
])

# ================== INITIAL STATE ==================
# Example 1: Start Sunny
initial_sunny = np.array([1.0, 0.0])

# Example 2: Start Rainy
initial_rainy = np.array([0.0, 1.0])


# ================== FUNCTION ==================
def next_step(state, matrix):
    return np.dot(state, matrix)


# ============================================================
# 🔥 SOLVED QUESTION (IN COMMENTS)
# ============================================================

"""
Q1: If today is Sunny, probability of Sunny after 2 days?

Step 1:
Day 0 = [1, 0]

Step 2:
Day 1 = [1,0] × matrix
      = [0.8, 0.2]

Step 3:
Day 2 = [0.8,0.2] × matrix

= Sunny:
  (0.8 × 0.8) + (0.2 × 0.4)
= 0.64 + 0.08
= 0.72

✔ Answer: 0.72


------------------------------------------------------------

Q2: If today is Rainy, probability of Sunny after 2 days?

Step 1:
Day 0 = [0, 1]

Step 2:
Day 1 = [0,1] × matrix
      = [0.4, 0.6]

Step 3:
Day 2 = [0.4,0.6] × matrix

= Sunny:
  (0.4 × 0.8) + (0.6 × 0.4)
= 0.32 + 0.24
= 0.56

✔ Answer: 0.56


------------------------------------------------------------

Q3: Transition Matrix is already given above:
[
 [0.8, 0.2],
 [0.4, 0.6]
]
"""


# ================== OPTIONAL SIMULATION ==================
state = initial_sunny

for i in range(2):
    state = next_step(state, transition_matrix)

print("After 2 steps (starting Sunny):", state)

In [ ]:
# MINIMAX 

# ================== SIMPLE MINIMAX ==================
import math

class Node:
    def __init__(self, value=None, children=None):
        self.value = value
        self.children = children or []


def minimax(node, depth, maximizing):

    # BASE CASE (leaf or depth limit)
    if depth == 0 or not node.children:
        return node.value

    # MAX PLAYER
    if maximizing:
        best = -math.inf
        for child in node.children:
            val = minimax(child, depth - 1, False)
            best = max(best, val)
        return best

    # MIN PLAYER
    else:
        best = math.inf
        for child in node.children:
            val = minimax(child, depth - 1, True)
            best = min(best, val)
        return best

In [ ]:
# MINMAX ALGO with Alpha-Beta pruning

import math

class Node:
    def __init__(self, value=None, children=None):
        self.value = value
        self.children = children or []


def alphabeta(node, depth, alpha, beta, maximizing):

    # BASE CASE
    if depth == 0 or not node.children:
        return node.value

    # ================== MAX PLAYER ==================
    if maximizing:
        best = -math.inf

        for child in node.children:
            val = alphabeta(child, depth - 1, alpha, beta, False)
            best = max(best, val)

            # UPDATE ALPHA
            alpha = max(alpha, best)

            # PRUNING CONDITION
            if beta <= alpha:
                break   # STOP exploring (prune)

        return best

    # ================== MIN PLAYER ==================
    else:
        best = math.inf

        for child in node.children:
            val = alphabeta(child, depth - 1, alpha, beta, True)
            best = min(best, val)

            # UPDATE BETA
            beta = min(beta, best)

            # PRUNING CONDITION
            if beta <= alpha:
                break   # STOP exploring (prune)

        return best

In [ ]:
# EDA + DATA PROCESSING

#EDA TEMPLATE

# =========================
# ML PREPROCESSING TEMPLATE
# Use for: classification / regression datasets
# =========================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler LabelEncoder

# 1. LOAD DATA
df = pd.read_csv("your_file.csv")

# 2. BASIC DATA CHECK
print(df.head())
print(df.info())
print(df.isnull().sum())

# 3. HANDLE MISSING VALUES
# Option 1: remove rows with missing values (simple but loses data)
df = df.dropna()

# Option 2 (alternative): fill missing numeric values with mean
# df = df.fillna(df.mean(numeric_only=True))

# 4. ENCODE CATEGORICAL COLUMNS (text → numbers)
# safer approach: one-hot encoding
df = pd.get_dummies(df, drop_first=True)

# 5. SPLIT FEATURES AND TARGET
target_column = "target_column"   # <-- change this

X = df.drop(target_column, axis=1)
y = df[target_column]

# 6. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 7. FEATURE SCALING (important for distance-based models)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)  # learn + transform
X_test = scaler.transform(X_test)        # only transform

# =========================
# DONE: READY FOR MODEL TRAINING
# =========================